# Phase 2: ML Model Training & Evaluation

This notebook trains LightGBM, Random Forest, and XGBoost on the Track A dataset.
It uses strict chronological splits to prevent data leakage and evaluates MAE, RMSE, R²,
and quantile pinball losses.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
import mlflow

from src.models.train_lightgbm import train_lgbm
from src.models.train_random_forest import train_rf
from src.models.train_xgboost import train_xgb
from src.models.data_split import chronological_split
from src.feature_engineering.feature_pipeline import build_feature_pipeline, save_pipeline

# Ensure MLflow UI tracks in local mlruns/
mlflow.set_tracking_uri("sqlite:///mlruns.db")

## 1. Fit & Save the Feature Pipeline
Before training the models, we fit the categorical encoders and other transforms.

In [ ]:
DATA_PATH = "../data/processed/food_delivery.parquet"

# First, fit the pipeline on the train split only to avoid leakage
df = pd.read_parquet(DATA_PATH)
train_df, _, _ = chronological_split(df)
X_train_raw = train_df.drop(columns=['actual_delivery_time_minutes', 'created_at'])

pipeline = build_feature_pipeline()
pipeline.fit(X_train_raw)

pipeline_path = save_pipeline(pipeline)
print(f"Pipeline saved to {pipeline_path}")

## 2. Train Models (MLflow)

In [ ]:
print("--- Training LightGBM ---")
train_lgbm(DATA_PATH, pipeline_path, "food_delivery")

print("\n--- Training Random Forest ---")
train_rf(DATA_PATH, pipeline_path, "food_delivery")

print("\n--- Training XGBoost ---")
train_xgb(DATA_PATH, pipeline_path, "food_delivery")

## 3. Interpretability (Feature Importance & SHAP)

In [ ]:
from src.models.model_registry import load_model
from src.models.feature_importance import compute_and_plot_importance
from src.models.explainability import plot_global_shap_summary

# Load test data properly
_, _, test_df = chronological_split(df)
X_test_raw = test_df.drop(columns=['actual_delivery_time_minutes', 'created_at'])
y_test = test_df['actual_delivery_time_minutes'].values
X_test = pipeline.transform(X_test_raw)

# Load best model
lgbm_p50, _ = load_model("lightgbm_p50")

compute_and_plot_importance(lgbm_p50, X_test, y_test, X_test.columns)
plot_global_shap_summary(lgbm_p50, X_test)

## Conclusion & R² Sanity Check
LightGBM with quantile objective is selected as the primary model due to native categorical handling, fast inference, and direct support for asymmetric quantile loss (crucial for p10/p90 uncertainty bounds).

**Sanity Check**: Any R² > 0.97 strongly suggests leakage. The implemented TimeSeriesSplit and chronological cutovers verified this is guarded against.